In [ ]:
import os.path
from pathlib import Path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

try:
    # Works when running as a .py script
    CREDENTIALS_DIR = Path(__file__).parent.parent / ".credentials"
except NameError:
    # Falls back when running in a Jupyter notebook
    CREDENTIALS_DIR = Path(os.getcwd()) / ".credentials"

TOKEN_JSON_PATH = CREDENTIALS_DIR / "show_labels_token.json"

In [ ]:
creds = None


# The file token.json stores the user's access and refresh tokens, and is
# created automatically when the authorization flow completes for the first
# time.
if os.path.exists(TOKEN_JSON_PATH):
    creds = Credentials.from_authorized_user_file(TOKEN_JSON_PATH, SCOPES)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
      flow = InstalledAppFlow.from_client_secrets_file(
          os.path.join(CREDENTIALS_DIR, "credentials.json"), SCOPES
      )
      creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open(TOKEN_JSON_PATH, "w") as token:
      token.write(creds.to_json())

try:
    # Call the Gmail API
    service = build("gmail", "v1", credentials=creds)
    results = service.users().labels().list(userId="me").execute()
    labels = results.get("labels", [])

    if not labels:
        print("No labels found.")
    else:
        print("Labels:")
        for label in labels:
            print(label["name"])

except HttpError as error:
    # TODO(developer) - Handle errors from gmail API.
    print(f"An error occurred: {error}")
